# mixedbench in a nutshell

Generalized linear mixed models (GLMMs) fit data that come in groups:
repeated measures per subject, pupils per school, plots per field.
Methods that do Bayesian inference on GLMMs are hard to compare,
since each paper tests them on its own handful of datasets.

mixedbench collects public grouped datasets, each with a model from its documentation.
It downloads each dataset at a fixed version and turns it into a problem ready to fit.

This notebook walks through one example: the classic [sleepstudy](https://rdrr.io/cran/lme4/man/sleepstudy.html),
reaction times of 18 subjects over ten days of restricted sleep.

On Colab or in a fresh environment, install mixedbench first:

In [ ]:
%pip install -q mixedbench

#### Overview:
1. Browse the registry
2. Load one table
3. Build one problem
4. Outlook

## 1. Browse the registry

The registry lists every dataset and model we found, one row per model.
Some tables have several models, so a row's id is `<source>/<table>/<model number>`.

In [2]:
import pandas as pd
from mixedbench import registry

pd.set_option('display.max_colwidth', None)
rows = registry.rows()
tables = {(row.source, row.object) for row in rows}
print(f'{len(rows)} models over {len(tables)} tables')
columns = ['id', 'response_type', 'formula']
records = [{column: getattr(row, column) for column in columns} for row in rows]
pd.DataFrame(records).head()

714 models over 641 tables


,id,response_type,formula
0,lme4/Arabidopsis/1,count,total.fruits ~ nutrient*amd + rack + status + (1|popu) + (1|gen)
1,lme4/Dyestuff/1,continuous,Yield ~ 1 + (1|Batch)
2,lme4/Pastes/1,continuous,strength ~ (1|batch/cask)
3,lme4/Penicillin/1,continuous,diameter ~ (1|plate) + (1|sample)
4,lme4/VerbAgg/1,binary,r2 ~ (Anger + Gender + btype + situ)^2 + (1|id) + (1|item)


## 2. Load one table

The loader downloads the table's source at a fixed version into `cache/`,
here the R package [lme4](https://rdrr.io/cran/lme4/) from CRAN.
It checks that the file is unchanged and returns the table with typed columns.

In [3]:
from pathlib import Path

cache = Path('cache')
[row] = [row for row in rows if row.id == 'lme4/sleepstudy/1']
print(row.formula)
table = registry.Tables(cache).load(row)
table.head()

Reaction ~ Days + (Days|Subject)


,Reaction,Days,Subject
0,249.5600,0.0,308
1,258.7047,1.0,308
2,250.8006,2.0,308
3,321.4398,3.0,308
4,356.8519,4.0,308


## 3. Build one problem

A *problem* is a row made ready to fit:
numeric columns, a model family, a prior, and held-out data to test predictions on.
The same fixed rules build every problem, so the results of two methods compare.

In [4]:
from mixedbench.contract import Problem, materialise

problem = Problem.build(row, table)
print('design: ', problem.design)
print('response:', problem.response, f'({problem.family}, {problem.link} link)')
print('factors: ', problem.factors)
problem.table.head()

design:  ('Intercept', 'Days')
response: Reaction (gaussian, identity link)
factors:  ('Subject',)


,Intercept,Days,Reaction,Subject,rows,groups
0,1.0,0.0,249.5600,0,seen,seen
1,1.0,1.0,258.7047,0,seen,seen
2,1.0,2.0,250.8006,0,seen,seen
3,1.0,3.0,321.4398,0,seen,seen
4,1.0,4.0,356.8519,0,seen,seen


The columns `rows` and `groups` mark each row `seen` (a fit sees it) or `held` (a fit must predict it).
A method is scored in one of two ways:
it fits all the data and its posterior is compared with a reference posterior,
or it fits the `seen` data and predicts the rest.
The held-out data are either new rows of known groups (`rows`) or whole new groups (`groups`).

Every problem uses the same default prior, set on standardised data.
The reference posteriors use it, so a method compared with them uses it too:

In [5]:
problem.prior

{'Intercept': Prior(distribution='normal', parameters=(0, 2.5)),
 'b': Prior(distribution='normal', parameters=(0, 2.5)),
 'sd': Prior(distribution='student_t', parameters=(3, 0, 2.5)),
 'cor': Prior(distribution='lkj', parameters=(1,)),
 'sigma': Prior(distribution='student_t', parameters=(3, 0, 2.5))}

`materialise` saves the problem to `cache/` and returns its hash, a fingerprint of every value:
the same hash means the same problem on any machine.
`load(id, cache)` reads it back later, offline.

In [6]:
materialise(problem, cache)

'8111b74ef393568a2cb445ef9f10cc4f538abd8d6a6d48c6b91181fac5a8f330'

## 4. Outlook

Release 1.1 adds the first *suite*: a fixed set of problems with their reference posteriors,
the metrics that score a method against them, and the scores of standard methods to compare with.
This notebook will be updated accordingly.
The plan is in the [roadmap](https://github.com/adkipnis/mixedbench/blob/main/ROADMAP.md).